# Game Board & Pattern Generator Toolkit

Making chessboards, Tic-Tac-Toe boards and other grid patterns with NumPy only.
Main things I'm practising: `np.zeros`, `np.ones`, `np.full`, `np.eye`, `np.arange`,
slicing with steps and fancy indexing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

## 1. Tic-Tac-Toe board

In [ ]:
# 0 = empty, 1 = X, 2 = O
board = np.zeros((3, 3), dtype=int)
print(board)
print("shape:", board.shape, "| dtype:", board.dtype)

In [ ]:
# same kind of board but built from a nested list
sample = np.array([[1, 0, 2],
                   [0, 1, 0],
                   [2, 0, 1]])
print(sample)

## 2. 8x8 chessboard using stepped slicing

In [ ]:
chess = np.zeros((8, 8), dtype=int)

chess[::2, ::2] = 1     # even rows, even columns
chess[1::2, 1::2] = 1   # odd rows, odd columns
print(chess)

plt.imshow(chess, cmap="gray")
plt.title("8x8 chessboard")
plt.xticks([])
plt.yticks([])
plt.show()

## 3. Other grids (ones, full, arange, checkerboard, identity)

In [ ]:
ones_grid = np.ones((4, 4), dtype=int)
sevens = np.full((4, 4), 7)
counting = np.arange(16).reshape(4, 4)

# checkerboard another way - add row and column numbers and look at odd/even
rows = np.arange(6).reshape(6, 1)
cols = np.arange(6)
checker = (rows + cols) % 2

identity = np.eye(5, dtype=int)

print("ones:\n", ones_grid)
print("full of 7:\n", sevens)
print("arange reshaped:\n", counting)
print("checkerboard:\n", checker)
print("identity:\n", identity)

## 4. Diagonal and anti-diagonal patterns

In [ ]:
diag = np.eye(5, dtype=int)
anti_diag = np.fliplr(diag)          # flip left-right, same as diag[:, ::-1]
both = np.maximum(diag, anti_diag)   # the X shape (max so the centre stays 1, not 2)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, grid, name in zip(axes, [diag, anti_diag, both],
                          ["diagonal", "anti-diagonal", "both (X)"]):
    ax.imshow(grid, cmap="Blues")
    ax.set_title(name)
    ax.axis("off")
plt.show()

## 5. Placing and moving pieces with fancy indexing

In [ ]:
board = np.zeros((3, 3), dtype=int)

# put X on the main diagonal by passing row and column index arrays
r = np.array([0, 1, 2])
c = np.array([0, 1, 2])
board[r, c] = 1

# two O's at once: (0,2) and (2,0)
board[[0, 2], [2, 0]] = 2
print(board)

In [ ]:
# move both O's to new squares in one go
old_r, old_c = np.array([0, 2]), np.array([2, 0])
new_r, new_c = np.array([1, 2]), np.array([2, 1])

pieces = board[old_r, old_c]   # grab what's on the old squares
board[old_r, old_c] = 0        # clear them
board[new_r, new_c] = pieces   # drop them on the new squares
print(board)

## 6. Win checker + showing the board

In [ ]:
def check_winner(b):
    """returns 1 if X wins, 2 if O wins, 0 if nobody has won yet"""
    n = b.shape[0]
    for player in (1, 2):
        # rows and columns
        for i in range(n):
            if np.all(b[i, :] == player) or np.all(b[:, i] == player):
                return player
        # main diagonal, then anti-diagonal (flip the columns first)
        if np.all(np.diag(b) == player) or np.all(np.diag(b[:, ::-1]) == player):
            return player
    return 0

In [ ]:
cmap = ListedColormap(["white", "salmon", "lightskyblue"])

def show_board(b, title=""):
    n = b.shape[0]
    plt.imshow(b, cmap=cmap, vmin=0, vmax=2)
    for (i, j), v in np.ndenumerate(b):
        if v == 1:
            plt.text(j, i, "X", ha="center", va="center", fontsize=22)
        elif v == 2:
            plt.text(j, i, "O", ha="center", va="center", fontsize=22)
    # grid lines between the cells
    for k in range(n - 1):
        plt.axhline(k + 0.5, color="black")
        plt.axvline(k + 0.5, color="black")
    plt.xticks([])
    plt.yticks([])
    plt.title(title)
    plt.show()

In [ ]:
# quick tests, one for each way to win plus a board with no winner
tests = {
    "row win (X)": np.array([[1, 1, 1], [2, 2, 0], [0, 0, 0]]),
    "column win (O)": np.array([[2, 1, 0], [2, 1, 0], [2, 0, 1]]),
    "diagonal win (X)": np.array([[1, 2, 0], [2, 1, 0], [0, 0, 1]]),
    "anti-diagonal win (O)": np.array([[1, 1, 2], [0, 2, 0], [2, 0, 1]]),
    "no winner yet": np.array([[1, 2, 1], [2, 1, 2], [2, 1, 2]]),
}

names = {0: "nobody", 1: "X", 2: "O"}
for name, b in tests.items():
    print(f"{name:25s} -> winner: {names[check_winner(b)]}")

In [ ]:
show_board(tests["anti-diagonal win (O)"], "O wins on the anti-diagonal")

## 7. Small demo game

In [ ]:
game = np.zeros((3, 3), dtype=int)

# (row, col) moves, X goes first and they alternate
moves = [(1, 1), (0, 0), (0, 2), (0, 1), (2, 0)]

for turn, (row, col) in enumerate(moves):
    player = 1 if turn % 2 == 0 else 2
    game[row, col] = player
    result = check_winner(game)
    if result != 0:
        print(f"{names[result]} wins after move {turn + 1}")
        break
else:
    print("no winner")

show_board(game, "final board")

## What I learned

- Stepped slicing (`[::2, 1::2]`) is a neat way to make alternating patterns without loops.
- `np.eye` plus `np.fliplr` gives both diagonals very quickly.
- Fancy indexing lets me place or move several pieces at once with two index arrays.
- Slicing makes the win checker short: one row, one column, and `np.diag` on the board and its flipped version.